In [1]:
import gc
import os
from catboost import CatBoostClassifier
import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score
import pickle

 ## Функция для оптимизации параметров с помощью optuna



In [8]:
from catboost import CatBoostClassifier, CatBoostError
import optuna
from sklearn.metrics import roc_auc_score

def objective(trial, X_train, y_train, X_val, y_val):
    # Гиперпараметры для CatBoost
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": trial.suggest_int("iterations", 100, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        
        "grow_policy": "Lossguide",
        "max_leaves": trial.suggest_int("max_leaves", 15, 255),
        "depth": trial.suggest_int("depth", 3, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 500),
        
        "bootstrap_type": "Bernoulli",
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        
        # ИСПРАВЛЕНИЕ 1: Увеличиваем нижний порог регуляризации до 1e-3 (было 1e-8)
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 19.0),
        "verbose": False,
        "thread_count": -1,
        "random_seed": 42
    }

    model = CatBoostClassifier(**params)

    # ИСПРАВЛЕНИЕ 2: Защита от падения C++ бэкенда (try-except)
    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=50,
            verbose=False
        )
    except CatBoostError as e:
        # Проверяем, та ли это ошибка
        if "This should be unreachable" in str(e):
            print(f"\n  [!] Пойман внутренний баг CatBoost на шаге {trial.number}. Пропускаем эти параметры...")
            # Приказываем Optuna отбросить эту итерацию (prune) и продолжить поиск
            raise optuna.exceptions.TrialPruned()
        else:
            # Если ошибка другая, пробрасываем её дальше
            raise e

    preds = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

## Настройка логирования

In [9]:

BASE_DIR = "../processed_data"
LOGS_DIR = "../catboost_optuna_log" # Изменил папку логов
os.makedirs(LOGS_DIR, exist_ok=True)

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Функция для логирования промежуточных шагов Optuna в файл
def logging_callback(study, trial):
    # Достаем имя набора из кастомных атрибутов исследования
    set_name = study.user_attrs.get("set_name", "unknown")
    log_file_path = os.path.join(LOGS_DIR, f"{set_name}_optimization.log")

    with open(log_file_path, "a") as f:
        # Записываем базовую информацию и параметры текущего шага
        f.write(
            f"Trial {trial.number:03d} finished | "
            f"ROC-AUC: {trial.value:.5f} | "
            f"Best ROC-AUC so far: {study.best_value:.5f} | "
            f"Params: {trial.params}\n"
        )
        
        # Если текущий шаг оказался лучшим, отдельно фиксируем это
        if trial.value == study.best_value:
            f.write(f"  [!] NEW BEST ROC-AUC! Best Params updated: {study.best_params}\n")

## Подбор параметров модели для каждого набора признаков (Полная обучающая выборка)

In [10]:
# Наборы признаков
feature_sets = [
    "top300",
    "top500_clean",
    "top500_lgb_clean",
    "top500_magic_meta",
    "top500_micro_engineered",
]

# Загружаем общие таргеты
y_train_full = (pd.read_parquet(os.path.join(BASE_DIR, "y_train_full.parquet")).iloc[:, 0].values.ravel())
y_val = (pd.read_parquet(os.path.join(BASE_DIR, "y_val.parquet")).iloc[:, 0].values.ravel())

# Словари для сохранения результатов
best_params_per_set = {}
best_scores_per_set = {}

# Главный цикл по наборам признаков
for folder in feature_sets:
    print(f"\n==================================================")
    print(f" НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: {folder}")
    print(f"==================================================")

    # Подгружаем только нужные матрицы признаков для Optuna
    x_train_full_path = os.path.join(BASE_DIR, folder, f"X_train_full_{folder}.parquet")
    x_val_path = os.path.join(BASE_DIR, folder, f"X_val_{folder}.parquet")

    X_train_full = pd.read_parquet(x_train_full_path)
    X_val = pd.read_parquet(x_val_path)

    # Инициализируем исследование Optuna
    study = optuna.create_study(direction="maximize")
    study.set_user_attr("set_name", folder)

    # Чистим старый лог-файл, если он остался от прошлых запусков
    log_file_path = os.path.join(LOGS_DIR, f"{folder}_optimization.log")
    if os.path.exists(log_file_path):
        os.remove(log_file_path)

    # Начинаем лог-файл с заголовка
    with open(log_file_path, "a") as f:
        f.write(f"=== Optimization Started for {folder} ===\n")

    # Запуск оптимизации
    study.optimize(
        lambda trial: objective(
            trial, X_train_full, y_train_full, X_val, y_val
        ),
        n_trials=50,  
        timeout=1800,
        callbacks=[logging_callback]
    )

    # Сохраняем лучшие результаты в словари
    best_params_per_set[folder] = study.best_params
    best_scores_per_set[folder] = study.best_value

    # --- НОВОЕ: Записываем финальные лучшие результаты в лог-файл ---
    with open(log_file_path, "a") as f:
        f.write(f"\n=== Optimization Finished for {folder} ===\n")
        f.write(f"FINAL BEST ROC-AUC: {study.best_value:.5f}\n")
        f.write("FINAL BEST PARAMS:\n")
        for param, value in study.best_params.items():
            f.write(f"   -> {param}: {value}\n")
    # ------------------------------------------------------------------

    # Выводим краткие итоги в консоль
    print(f"\n УСПЕШНО ЗАВЕРШЕНО ДЛЯ: {folder}")
    print(f" Лучший ROC-AUC на валидации: {study.best_value:.5f}")
    print(f" Подробный лог сохранен в: {log_file_path}")
    print(f" Лучшие параметры:")
    for param, value in study.best_params.items():
        print(f"   -> {param}: {value}")

    # Очистка памяти перед следующей итерацией
    del X_train_full, X_val
    gc.collect()

print("\n\n==================================================")
print(" ОПТИМИЗАЦИЯ ВСЕХ НАБОРОВ ЗАВЕРШЕНА!")
print("==================================================")


 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top300

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top300
 Лучший ROC-AUC на валидации: 0.64797
 Подробный лог сохранен в: ../catboost_optuna_log\top300_optimization.log
 Лучшие параметры:
   -> iterations: 1345
   -> learning_rate: 0.022177504608059728
   -> max_leaves: 90
   -> depth: 5
   -> min_data_in_leaf: 425
   -> subsample: 0.8817131597529161
   -> colsample_bylevel: 0.5025121329459716
   -> l2_leaf_reg: 0.17792693514201682
   -> scale_pos_weight: 2.3446016834814394

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_clean

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_clean
 Лучший ROC-AUC на валидации: 0.66057
 Подробный лог сохранен в: ../catboost_optuna_log\top500_clean_optimization.log
 Лучшие параметры:
   -> iterations: 1349
   -> learning_rate: 0.04588961939978892
   -> max_leaves: 82
   -> depth: 10
   -> min_data_in_leaf: 210
   -> subsample: 0.6834139270804198
   -> colsample_bylevel: 0.8676605683187681
   -> l2_leaf_reg: 0.18988911197228484
   -> scale_pos_weight: 3.6110457435559296

 

Training has stopped (degenerate solution on iteration 251, probably too small l2-regularization, try to increase it)



 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_lgb_clean
 Лучший ROC-AUC на валидации: 0.66290
 Подробный лог сохранен в: ../catboost_optuna_log\top500_lgb_clean_optimization.log
 Лучшие параметры:
   -> iterations: 1289
   -> learning_rate: 0.040343066391463664
   -> max_leaves: 171
   -> depth: 7
   -> min_data_in_leaf: 265
   -> subsample: 0.6500363807158793
   -> colsample_bylevel: 0.636361108575674
   -> l2_leaf_reg: 0.3755263837947061
   -> scale_pos_weight: 5.4889274771431396

 НАЧАЛО ОПТИМИЗАЦИИ НАБОРА: top500_magic_meta

 УСПЕШНО ЗАВЕРШЕНО ДЛЯ: top500_magic_meta
 Лучший ROC-AUC на валидации: 0.65727
 Подробный лог сохранен в: ../catboost_optuna_log\top500_magic_meta_optimization.log
 Лучшие параметры:
   -> iterations: 724
   -> learning_rate: 0.041613144751384404
   -> max_leaves: 255
   -> depth: 7
   -> min_data_in_leaf: 298
   -> subsample: 0.5010698673525462
   -> colsample_bylevel: 0.7562670074207688
   -> l2_leaf_reg: 0.5124083282548205
   -> scale_pos_weight: 13.390676829953438

 НАЧ

## Результаты оптимизации с обучением на полной обучающей выборке

In [11]:
for folder in feature_sets:
    print(f"Dataset: {folder:<25} | Best ROC-AUC: {best_scores_per_set[folder]:.5f}")

Dataset: top300                    | Best ROC-AUC: 0.64797
Dataset: top500_clean              | Best ROC-AUC: 0.66057
Dataset: top500_lgb_clean          | Best ROC-AUC: 0.66290
Dataset: top500_magic_meta         | Best ROC-AUC: 0.65727
Dataset: top500_micro_engineered   | Best ROC-AUC: 0.65279


 ## Финальное обучение моделей с лучшими параметрами и их сохранение

In [ ]:
# [Code]

# Настройки путей
BASE_DIR = "../processed_data"
MODELS_DIR = "../models/catboost_models"  # Папка для моделей CatBoost
os.makedirs(MODELS_DIR, exist_ok=True)

y_train_full = (pd.read_parquet(os.path.join(BASE_DIR, "y_train_full.parquet")).iloc[:, 0].values.ravel())
y_val = (pd.read_parquet(os.path.join(BASE_DIR, "y_val.parquet")).iloc[:, 0].values.ravel())

final_val_scores = {}

for folder, params in best_params_per_set.items():
    print(f"\n==================================================")
    print(f" ФИНАЛЬНОЕ ОБУЧЕНИЕ НА ПОЛНОМ ТРЕЙНЕ: {folder}")
    print(f"==================================================")
    
    # Загружаем полные матрицы признаков для текущего датасета
    X_train_full = pd.read_parquet(os.path.join(BASE_DIR, folder, f"X_train_full_{folder}.parquet"))
    X_val = pd.read_parquet(os.path.join(BASE_DIR, folder, f"X_val_{folder}.parquet"))

    # Копируем параметры, найденные Optuna
    final_params = params.copy()
    
    # Жестко прописываем технические параметры и исправленную архитектуру деревьев
    final_params["loss_function"] = "Logloss"
    final_params["eval_metric"] = "AUC"
    
    # КРИТИЧЕСКИ ВАЖНО: дублируем параметры, которые лечат ошибку "This should be unreachable"
    final_params["grow_policy"] = "Lossguide"
    final_params["bootstrap_type"] = "Bernoulli"
    
    final_params["verbose"] = False
    final_params["thread_count"] = -1
    final_params["random_seed"] = 42

    # Инициализируем финальную модель
    final_model = CatBoostClassifier(**final_params)

    # Обучаем модель с контролем переобучения (early stopping)
    final_model.fit(
        X_train_full,
        y_train_full,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=50,
        verbose=False
    )

    # Считаем итоговый скор на валидации
    val_preds = final_model.predict_proba(X_val)[:, 1]
    current_score = roc_auc_score(y_val, val_preds)
    final_val_scores[folder] = current_score

    print(f"Проверочный ROC-AUC на валидации: {current_score:.5f}")

    # Сохраняем готовую модель в формате .pkl
    model_filename = os.path.join(MODELS_DIR, f"cb_{folder}_{current_score:.5f}.pkl")
    with open(model_filename, "wb") as f:
        pickle.dump(final_model, f)
    print(f"Модель успешно сохранена: {model_filename}")

    # Очищаем оперативную память перед следующим датасетом
    del X_train_full, X_val, final_model
    gc.collect()

print("\n\n==================================================")
print(" ОБУЧЕНИЕ И ИМПОРТ ВСЕХ МОДЕЛЕЙ ЗАВЕРШЕНЫ!")
print("==================================================")


 ФИНАЛЬНОЕ ОБУЧЕНИЕ ЛУЧШЕГО НАБОРА: top500_lgb_clean


FileNotFoundError: [Errno 2] No such file or directory: '/home/jupyter/project/processed_data\\y_train_full.parquet'